### Parent Document Retriever (ParentDocumentRetriever)

When splitting text for RAG, developers face a tradeoff:
* Small chunks are better for precise vector embedding matches.
* Large chunks provide richer contextual information for LLM response generation.

ParentDocumentRetriever resolves this by indexing small child chunks in a vector store for search, while returning the full parent document from an underlying key-value store.

### Architecture & Workflow

1. Parent Splitter: Splits raw documents into large parent chunks.
2. Child Splitter: Splits parent chunks into small child sub-chunks.
3. VectorStore: Stores and indexes child sub-chunk vectors.
4. DocStore: Stores complete parent chunks in an InMemoryStore keyed by parent document ID.

In [1]:
from dotenv import load_dotenv, find_dotenv
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.stores import InMemoryStore
from langchain_core.documents import Document

load_dotenv(find_dotenv())

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
# Ephemeral in-memory Chroma collection
vectorstore = Chroma(collection_name="split_parents", embedding_function=embeddings)
store = InMemoryStore()

# Define parent and child splitters
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=1000)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=200)

# Initialize ParentDocumentRetriever
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)

large_doc = Document(
    page_content="""The Apollo 11 mission was the spaceflight that first landed humans on the Moon.
Commander Neil Armstrong and Lunar Module Pilot Buzz Aldrin landed the Apollo Lunar Module Eagle on July 20, 1969.
Armstrong became the first person to step onto the lunar surface six hours and 39 minutes later on July 21.
Aldrin joined him 19 minutes later. They spent two and a quarter hours together outside the spacecraft.
They collected 47.5 pounds (21.5 kg) of lunar material to bring back to Earth.
Command Module Pilot Michael Collins flew the Command Module Columbia alone in lunar orbit while they were on the surface.
Armstrong and Aldrin spent 21.5 hours on the lunar surface at a site they named Tranquility Base before lifting off to rejoin Columbia.
"""
)

# Add document to ParentDocumentRetriever
retriever.add_documents([large_doc])

# Search query matching a small child chunk
retrieved_parents = retriever.invoke("How much lunar material was collected?")

print(f"Retrieved {len(retrieved_parents)} Parent Document(s).")
print(f"Parent Document Length: {len(retrieved_parents[0].page_content)} characters")
print(f"Parent Content Preview:\n{retrieved_parents[0].page_content[:300]}...")


/Users/kapilyadav/Coding_Space/Python_workspace/LangChainWorkspace/generativeai/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Retrieved 1 Parent Document(s).
Parent Document Length: 744 characters
Parent Content Preview:
The Apollo 11 mission was the spaceflight that first landed humans on the Moon.
Commander Neil Armstrong and Lunar Module Pilot Buzz Aldrin landed the Apollo Lunar Module Eagle on July 20, 1969.
Armstrong became the first person to step onto the lunar surface six hours and 39 minutes later on July 2...
